# Gold Layer

Builds the business-ready Gold layer from the cleaned Silver tables: a calendar-enriched daily revenue summary, a customer-level performance table, and a consolidated category league table. Each gold table is written to `workspace.gold` and reconciled against the silver totals below.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, when, trim, coalesce, lit
from pyspark.sql.window import Window

## Reading from the Silver Tables

**Purpose:** Loads the three cleaned Silver tables that feed the Gold layer, and creates the `workspace.gold` schema if it does not yet exist.

**Sources:**
- `workspace.silver.silver_revenue_data` — 1,097 daily revenue rows
- `workspace.silver.silver_orders_data` — 1,577 order-level rows
- `workspace.silver.silver_categories_data` — 238 category rows (232 distinct categories)

In [0]:
# Ensure the gold schema exists before the first write
spark.sql('CREATE SCHEMA IF NOT EXISTS workspace.gold')

revenue_df = spark.table('workspace.silver.silver_revenue_data')
orders_df = spark.table('workspace.silver.silver_orders_data')
categories_df = spark.table('workspace.silver.silver_categories_data')

print(f"silver_revenue_data: {revenue_df.count()} daily rows")
print(f"silver_orders_data: {orders_df.count()} orders")
print(f"silver_categories_data: {categories_df.count()} category rows")

## Data Transformations

**Purpose:** Builds three business-ready gold tables, one cell per table:

1. **Gold Daily Revenue Summary** — the daily revenue enriched with calendar attributes for reporting and time analysis
2. **Gold Customer Summary** — one row per customer with lifetime performance and their most-used acquisition channel
3. **Gold Category Performance** — a consolidated category league table with revenue rankings and Pareto shares

### 1. Gold Daily Revenue Summary

One row per day from the silver revenue table, enriched with year, quarter, month, month name, day of week, and a weekend flag — so dashboards and analysts can slice revenue by any calendar dimension without recomputing it.

In [0]:
gold_daily_df = (revenue_df
    .withColumn('year', F.year('Date'))
    .withColumn('quarter', F.concat(F.col('year').cast('string'), F.lit('-Q'), F.quarter('Date').cast('string')))
    .withColumn('month', F.to_date(F.date_trunc('month', col('Date'))))
    .withColumn('month_name', F.date_format('Date', 'MMMM'))
    .withColumn('day_of_week', F.date_format('Date', 'EEEE'))
    .withColumn('is_weekend', F.dayofweek('Date').isin(1, 7))
    .select(
        col('Date').alias('date'),
        'year', 'quarter', 'month', 'month_name', 'day_of_week', 'is_weekend',
        col('Orders').alias('orders'),
        col('Gross_sales').alias('gross_sales'),
        col('Returns').alias('returns'),
        col('Coupons').alias('coupons'),
        col('Net_sales').alias('net_sales'),
        col('Taxes').alias('taxes'),
        col('Shipping').alias('shipping'),
        col('Total_sales').alias('total_sales'),
    )
)

gold_daily_df.write.mode('overwrite').option('overwriteSchema', 'true') \
    .saveAsTable('workspace.gold.gold_daily_revenue_summary')

print(f"\u2713 gold_daily_revenue_summary: {gold_daily_df.count()} rows")
display(gold_daily_df.orderBy(F.col('date').desc()).limit(10))

### 2. Gold Customer Summary

One row per customer: their current type (the type on their most recent order — a customer's type can change from New to Returning over their history), lifetime orders, revenue, average order value, items bought, savings and coupon discounts granted, refunded orders, first and last order dates, active span in days, and the acquisition channel they order through most often.

In [0]:
# Most-used acquisition channel per customer (most orders, ties broken alphabetically)
top_channel_df = (orders_df
    .groupBy('customer_name', 'attribution')
    .agg(F.count('*').alias('channel_orders'))
    .withColumn('rn', F.row_number().over(
        Window.partitionBy('customer_name').orderBy(F.col('channel_orders').desc(), F.col('attribution').asc())))
    .filter(F.col('rn') == 1)
    .select('customer_name', F.col('attribution').alias('top_attribution_channel'))
)

# A customer's type can change over their history (New → Returning), so take the
# type from their most recent order to guarantee one row per customer
latest_type_df = (orders_df
    .withColumn('rn', F.row_number().over(
        Window.partitionBy('customer_name').orderBy(F.col('order_date').desc(), F.col('order_number').desc())))
    .filter(F.col('rn') == 1)
    .select('customer_name', F.col('customer_type').alias('current_customer_type'))
)

gold_customers_df = (orders_df
    .groupBy('customer_name')
    .agg(
        F.count('*').alias('orders'),
        F.round(F.sum('revenue_amount'), 2).alias('total_revenue'),
        F.round(F.avg('revenue_amount'), 2).alias('avg_order_value'),
        F.sum('items_sold').alias('items_sold'),
        F.round(F.sum('savings_amount'), 2).alias('total_savings'),
        F.sum('coupon_amount').alias('coupon_discounts'),
        F.sum(when(F.col('order_status') == 'Refunded', 1).otherwise(0)).alias('refunded_orders'),
        F.min('order_date').alias('first_order'),
        F.max('order_date').alias('last_order'),
        F.datediff(F.max('order_date'), F.min('order_date')).alias('active_days'),
    )
    .join(latest_type_df, on='customer_name', how='left')
    .join(top_channel_df, on='customer_name', how='left')
)

gold_customers_df.write.mode('overwrite').option('overwriteSchema', 'true') \
    .saveAsTable('workspace.gold.gold_customer_summary')

print(f"\u2713 gold_customer_summary: {gold_customers_df.count()} customers")
display(gold_customers_df.orderBy(F.col('total_revenue').desc()).limit(10))

### 3. Gold Category Performance

A league table consolidating the 232 distinct categories (merging the duplicate and near-duplicate names held in the silver table), with each category's revenue rank, its share of total net revenue, and the cumulative share that shows how concentrated revenue is.

In [0]:
# Consolidate duplicate category names, then rank and accumulate revenue share
category_base_df = (categories_df
    .groupBy('Category')
    .agg(
        F.sum('Items_sold').alias('items_sold'),
        F.round(F.sum('Net_Revenue'), 2).alias('net_revenue'),
        F.sum('Products').alias('products'),
        F.sum('Orders').alias('orders'),
    )
)

gold_categories_df = (category_base_df
    .withColumn('revenue_rank', F.row_number().over(Window.orderBy(F.col('net_revenue').desc())))
    .withColumn('pct_of_total_revenue', F.round(
        F.col('net_revenue') / F.sum('net_revenue').over(Window.orderBy(F.lit(1))) * 100, 2))
    .withColumn('cum_pct_of_revenue', F.round(
        F.sum('net_revenue').over(Window.orderBy('revenue_rank').rowsBetween(Window.unboundedPreceding, Window.currentRow))
        / F.sum('net_revenue').over(Window.orderBy(F.lit(1))) * 100, 2))
)

gold_categories_df.write.mode('overwrite').option('overwriteSchema', 'true') \
    .saveAsTable('workspace.gold.gold_category_performance')

print(f"\u2713 gold_category_performance: {gold_categories_df.count()} categories")
display(gold_categories_df.orderBy('revenue_rank').limit(10))

## Verification

**Purpose:** Reconciles every gold table against its silver source — row counts and revenue totals must match, proving the gold layer lost or duplicated nothing on the way up.

In [0]:
reconciliation = spark.sql("""
    SELECT
        (SELECT COUNT(*) FROM workspace.gold.gold_daily_revenue_summary) AS gold_daily_rows,
        (SELECT COUNT(*) FROM workspace.silver.silver_revenue_data) AS silver_revenue_rows,
        (SELECT ROUND(SUM(net_sales), 2) FROM workspace.gold.gold_daily_revenue_summary) AS gold_daily_net,
        (SELECT ROUND(SUM(net_sales), 2) FROM workspace.silver.silver_revenue_data) AS silver_revenue_net,
        (SELECT COUNT(*) FROM workspace.gold.gold_customer_summary) AS gold_customer_rows,
        (SELECT COUNT(DISTINCT customer_name) FROM workspace.silver.silver_orders_data) AS silver_distinct_customers,
        (SELECT ROUND(SUM(total_revenue), 2) FROM workspace.gold.gold_customer_summary) AS gold_customer_revenue,
        (SELECT ROUND(SUM(revenue_amount), 2) FROM workspace.silver.silver_orders_data) AS silver_orders_revenue,
        (SELECT COUNT(*) FROM workspace.gold.gold_category_performance) AS gold_category_rows,
        (SELECT ROUND(SUM(net_revenue), 2) FROM workspace.gold.gold_category_performance) AS gold_category_net,
        (SELECT ROUND(SUM(Net_Revenue), 2) FROM workspace.silver.silver_categories_data) AS silver_category_net
""")

display(reconciliation)